# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ashoktanakanti/flyrank_ml/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

**The rule, in plain words:** a page is worth reviewing first if it is (a) getting real
search demand right now, (b) inconsistent/thin on visibility across the month, and (c) sits
in a position where a small improvement could matter. I combine these three signals into one
0–100 score — no ML, just a transparent weighted formula anyone can audit.

Built on `fact_content_daily_performance` (March 2026, same slice as ML-04/ML-06), aggregated
to one row per page. This table has no `days_since_last_update` or `word_count`, so I use
`days_with_data` (how many days in the month the page had any recorded data) as my staleness/
consistency proxy instead.

**Reason codes the rule can output:**
- `low_consistency_visible_page` — few days with data this month but still real impressions
- `low_ctr_visible_page` — good position but weak CTR, still real impressions
- `declining_with_demand` — currently declining (second-half clicks < first-half clicks) and still has demand
- `general_refresh_review` — flagged by the score but none of the above specific reasons fired

In [ ]:
import os
import pandas as pd
import numpy as np
from datasets import load_dataset
from huggingface_hub import login, notebook_login

HF_TOKEN = os.environ.get("HF_TOKEN")
if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
else:
    notebook_login()
    HF_TOKEN = os.environ.get("HF_TOKEN")

data_files = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"
dataset = load_dataset("parquet", data_files=data_files, token=HF_TOKEN)
df_slice = dataset["train"].to_pandas()
df_slice["report_date"] = pd.to_datetime(df_slice["report_date"])

print(f"Loaded {len(df_slice):,} rows for March 2026")

# Aggregate to one row per page
page = df_slice.groupby("content_hash_id").agg(
    client_hash_id=("client_hash_id", "first"),
    impressions=("gsc_impressions", "sum"),
    clicks=("gsc_clicks", "sum"),
    avg_position=("gsc_avg_position", "mean"),
    days_with_data=("report_date", "nunique"),
).reset_index()
page["ctr"] = np.where(page["impressions"] > 0, page["clicks"] / page["impressions"], 0)

# Declining label (same first-half vs second-half logic as ML-04/ML-06)
median_date = df_slice["report_date"].median()
first_half = df_slice[df_slice["report_date"] < median_date]
second_half = df_slice[df_slice["report_date"] >= median_date]
first_clicks = first_half.groupby("content_hash_id")["gsc_clicks"].sum()
second_clicks = second_half.groupby("content_hash_id")["gsc_clicks"].sum()
page["first_half_clicks"] = page["content_hash_id"].map(first_clicks).fillna(0)
page["second_half_clicks"] = page["content_hash_id"].map(second_clicks).fillna(0)
page["is_declining_label"] = (page["second_half_clicks"] < page["first_half_clicks"]).astype(int)

print(f"Pages: {len(page):,} | Declining rate: {page['is_declining_label'].mean():.1%}")
display(page[["impressions", "clicks", "avg_position", "ctr", "days_with_data"]].describe().round(2))

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

Loaded 9,841,378 rows for March 2026
Pages: 331,437 | Declining rate: 8.7%


,impressions,clicks,avg_position,ctr,days_with_data
count,331437.00,331437.00,176738.00,331437.00,331437.00
mean,846.79,2.48,16.00,0.00,29.69
std,4044.51,19.65,17.69,0.03,4.74
min,0.00,0.00,0.00,0.00,1.00
25%,0.00,0.00,5.00,0.00,31.00
50%,2.00,0.00,8.51,0.00,31.00
75%,216.00,0.00,20.37,0.00,31.00
max,617124.00,5668.00,309.00,1.00,31.00


## 2. Build the ranked queue (writes the CSV)

I score every page 0–100 from three percentile-ranked signals (demand, low consistency,
position opportunity), tag each with reason codes, sort descending, and save the queue.

In [ ]:
# 1. Three simple signals, each turned into a 0-1 percentile rank
demand = page["impressions"].rank(pct=True)
low_consistency = (1 - page["days_with_data"].rank(pct=True))  # fewer days = higher score

has_position = page["avg_position"] > 0
position_opportunity = pd.Series(0.0, index=page.index)
position_opportunity[has_position] = (1 - page.loc[has_position, "avg_position"].clip(upper=50) / 50).rank(pct=True)

# 2. Combine into one score (weights sum to 1, my own judgment call)
page["baseline_score"] = (
    0.45 * demand
    + 0.35 * low_consistency
    + 0.20 * position_opportunity
) * 100

# 3. Reason codes
median_days = page["days_with_data"].median()

def reason_codes(row):
    reasons = []
    if row["days_with_data"] <= median_days and row["impressions"] >= 500:
        reasons.append("low_consistency_visible_page")
    if row["avg_position"] > 0 and row["avg_position"] <= 20 and row["ctr"] < 0.02 and row["impressions"] >= 500:
        reasons.append("low_ctr_visible_page")
    if row["is_declining_label"] == 1 and row["impressions"] >= 100:
        reasons.append("declining_with_demand")
    return "|".join(reasons) if reasons else "general_refresh_review"

page["reason_codes"] = page.apply(reason_codes, axis=1)

# 4. Rank and save
ranked_queue = page.sort_values("baseline_score", ascending=False).reset_index(drop=True)
ranked_queue["rank"] = ranked_queue.index + 1

os.makedirs("work/outputs", exist_ok=True)
out_cols = ["rank", "content_hash_id", "client_hash_id", "baseline_score", "reason_codes",
            "impressions", "clicks", "avg_position", "ctr", "days_with_data", "is_declining_label"]
ranked_queue[out_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)

print(f"Wrote {len(ranked_queue):,} ranked rows to work/outputs/baseline_action_score.csv")
print(f"Top score: {ranked_queue['baseline_score'].max():.1f} | Median score: {ranked_queue['baseline_score'].median():.1f}")

Wrote 331,437 ranked rows to work/outputs/baseline_action_score.csv
Top score: 97.7 | Median score: 45.2


## 3. Top-20 review

The top 20 by score, with the reason code, and how many actually turned out to be declining
(a rough directional check — not a validated metric yet, that comes in Week 6).

In [ ]:
top_20 = ranked_queue.head(20)
display(top_20[["rank", "baseline_score", "reason_codes", "impressions",
                 "avg_position", "days_with_data", "is_declining_label"]])

print(f"\nOf the top 20, {top_20['is_declining_label'].sum()} of 20 are actually labeled declining "
      f"({top_20['is_declining_label'].mean():.0%}).")
print("Confidence note: this is a directional, decision-support signal on one month's warehouse "
      "slice, not a guarantee for any single page.")

,rank,baseline_score,reason_codes,impressions,avg_position,days_with_data,is_declining_label
0,1,97.694662,low_consistency_visible_page|low_ctr_visible_page,18722,1.986176,18,0
1,2,97.631857,low_consistency_visible_page|low_ctr_visible_page,28204,1.870249,20,0
2,3,97.499553,low_consistency_visible_page|low_ctr_visible_page,35480,2.379890,18,0
3,4,97.211855,low_consistency_visible_page|low_ctr_visible_page,7864,0.692351,20,0
4,5,97.086483,low_consistency_visible_page|low_ctr_visible_page,8061,2.115086,13,0
5,6,97.029197,low_consistency_visible_page|low_ctr_visible_page,11095,1.978689,20,0
6,7,96.983927,low_consistency_visible_page|low_ctr_visible_page,8795,1.615915,20,0
7,8,96.846253,low_consistency_visible_page|low_ctr_visible_page,13032,2.571033,18,0
8,9,96.640000,low_consistency_visible_page|low_ctr_visible_page,10567,1.998026,24,0
9,10,96.574695,low_consistency_visible_page|low_ctr_visible_page,7186,2.154665,18,0



Of the top 20, 0 of 20 are actually labeled declining (0%).
Confidence note: this is a directional, decision-support signal on one month's warehouse slice, not a guarantee for any single page.


## 4. Weak picks + leakage check

**Weak picks:** the highest-scoring pages that turned out NOT to be declining — where my rule
was wrong, and why.

**Leakage check:** confirming my score only uses observable, whole-month signals (impressions,
days_with_data, avg_position, ctr) — never the second-half-of-month clicks that define the
label, and never any FlyRank product flag.

In [ ]:
weak_picks = ranked_queue[ranked_queue["is_declining_label"] == 0].head(10)
print("=== TOP WEAK PICKS (high score, but not actually declining) ===")
display(weak_picks[["rank", "baseline_score", "reason_codes", "impressions",
                     "avg_position", "days_with_data"]])

score_inputs = ["impressions", "days_with_data", "avg_position", "ctr"]
print("\nColumns used to build baseline_score:", score_inputs)
print("second_half_clicks / first_half_clicks used only to LABEL rows for this review, "
      "never as score inputs: confirmed.")

=== TOP WEAK PICKS (high score, but not actually declining) ===


,rank,baseline_score,reason_codes,impressions,avg_position,days_with_data
0,1,97.694662,low_consistency_visible_page|low_ctr_visible_page,18722,1.986176,18
1,2,97.631857,low_consistency_visible_page|low_ctr_visible_page,28204,1.870249,20
2,3,97.499553,low_consistency_visible_page|low_ctr_visible_page,35480,2.379890,18
3,4,97.211855,low_consistency_visible_page|low_ctr_visible_page,7864,0.692351,20
4,5,97.086483,low_consistency_visible_page|low_ctr_visible_page,8061,2.115086,13
5,6,97.029197,low_consistency_visible_page|low_ctr_visible_page,11095,1.978689,20
6,7,96.983927,low_consistency_visible_page|low_ctr_visible_page,8795,1.615915,20
7,8,96.846253,low_consistency_visible_page|low_ctr_visible_page,13032,2.571033,18
8,9,96.640000,low_consistency_visible_page|low_ctr_visible_page,10567,1.998026,24
9,10,96.574695,low_consistency_visible_page|low_ctr_visible_page,7186,2.154665,18



Columns used to build baseline_score: ['impressions', 'days_with_data', 'avg_position', 'ctr']
second_half_clicks / first_half_clicks used only to LABEL rows for this review, never as score inputs: confirmed.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.